# S6E10 | TabPFN-3.5 on Encodings (candidate members)

In the meta-stack, the tabular foundation models carry the largest weights (TabFM 0.223, a 15-fold TabPFN-3.5 0.199,
LimiX-2 0.172), and every one of them was trained by someone else on the raw columns. This notebook gives TabPFN-3.5
our own features instead: the raw columns plus the original-data model (`orig_pred`), its per-value rates
(`orig_te`), value counts (`freq`) and the fold-safe target encodings (`te`, `te_pair`). The meta-stack notebook
screens each member on top of the full stack and keeps only those that help.

The features, folds and original-data model are exactly those of the Encodings + CatBoost + RealMLP notebook
(`StratifiedKFold(5, shuffle=True, random_state=42)`).

**Settings**
- Accelerator **GPU T4 x2** (the two GPUs fit two folds at a time) and **Internet on** (installs `tabpfn` and
  downloads the weights).
- Inputs: the competition data and the original survey, as in the Encodings notebook.
- **Add-ons → Secrets:** `TABPFN_TOKEN`, an API key from
  [platform.priorlabs.ai/account/api-keys](https://platform.priorlabs.ai/account/api-keys), on the account that
  accepted the TabPFN-3.5 licence. The weights download only with it; TabPFN-3.5 is under Prior Labs' non-commercial
  licence.
- Run with `SMOKE = True` first: one full-size fold per GPU, with timings and a projection of the full run. Then
  `SMOKE = False`.

In [ ]:
import os
import subprocess
import sys

# TabPFN-3.5 is the default model of tabpfn 9.x; its weights download on first use once the licence token is set
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'tabpfn>=9'], check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['TABPFN_TOKEN'] = UserSecretsClient().get_secret('TABPFN_TOKEN')
except Exception as e:                      # outside Kaggle, or no secret attached: use the environment as it is
    print(f'Kaggle secret TABPFN_TOKEN not read ({type(e).__name__})')
os.environ['TABPFN_NO_BROWSER'] = '1'       # never wait for a browser login in a batch run
print('TABPFN_TOKEN set:', bool(os.environ.get('TABPFN_TOKEN')))

In [ ]:
import gc, glob, queue, time, warnings
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
import lightgbm as lgb
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from tabpfn import TabPFNClassifier
warnings.filterwarnings('ignore')

SEED, TARGET, N_SPLITS = 42, 'satisfaction', 5
SMOKE = True      # True: one full-size fold per GPU (test cut to 20k rows), with timings and a projection of the full run
N_GPU = torch.cuda.device_count()
OUT = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
assert N_GPU > 0, 'no GPU: Settings -> Accelerator -> GPU T4 x2'
assert int(version('tabpfn').split('.')[0]) >= 9, f'tabpfn {version("tabpfn")}: TabPFN-3.5 needs tabpfn 9 or later'
print(f'tabpfn {version("tabpfn")} | torch {torch.__version__} | {N_GPU} x {torch.cuda.get_device_name(0)} | '
      f'smoke run: {SMOKE}')

## 1. Data, preprocessing and features

Unchanged from the Encodings + CatBoost + RealMLP notebook: the original-data model (`orig_pred`), its per-value
rates (`orig_te`), value counts (`freq`), and the fold-safe target encodings (`te`, `te_pair`) built once per fold.

The smoke run uses every row: TabPFN's time and memory grow with the training fold, so a sample would not show
whether the full run fits.

In [ ]:
def find(name, hint):
    hits = sorted(p for p in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if hint in p)
    assert hits, f'{name} ({hint}) not found under /kaggle/input'
    return hits[0]


KAGGLE = os.path.exists('/kaggle/input')
train_df = pd.read_csv(find('train.csv', 'playground-series-s6e10') if KAGGLE else '../data/train.csv')
test_df = pd.read_csv(find('test.csv', 'playground-series-s6e10') if KAGGLE else '../data/test.csv')
FEATURES = [c for c in test_df.columns if c != 'id']
CAT_COLS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
CONT_COLS = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
RATINGS = [c for c in FEATURES if c not in CAT_COLS + CONT_COLS]       # the 13 service ratings, 0-5
NUM_COLS = CONT_COLS + RATINGS


def load_original():
    """The original survey: any attached CSV with the competition's columns and a satisfaction label.
    Works with the 129,880-row single file and with the train/test pair of the original Kaggle dataset."""
    paths = (glob.glob('/kaggle/input/**/*.csv', recursive=True) if KAGGLE else ['../data/original_data.csv'])
    frames = []
    for path in sorted(p for p in paths if 'playground-series-s6e10' not in p):
        cols = pd.read_csv(path, nrows=0).columns
        if TARGET in cols and set(FEATURES) <= set(cols):
            frames.append(pd.read_csv(path, usecols=FEATURES + [TARGET]))
    assert frames, 'attach the original Airline Passenger Satisfaction dataset as an input'
    orig = pd.concat(frames, ignore_index=True).drop_duplicates(ignore_index=True)
    orig[TARGET] = orig[TARGET].astype(str).str.strip().str.lower().isin(['true', 'satisfied', '1']).astype('int8')
    return orig


orig_df = load_original()
train_df[TARGET] = train_df[TARGET].astype(str).str.strip().str.lower().eq('true').astype('int8')
y, y_orig = train_df[TARGET].values, orig_df[TARGET].values
print(f'train {train_df.shape} | test {test_df.shape} | original {orig_df.shape} | '
      f'positive rate {y.mean():.4f} (original {y_orig.mean():.4f})')

In [ ]:
CAT_MAPS = {
    'Gender': {'Female': 0, 'Male': 1},
    'Customer Type': {'disloyal Customer': 0, 'Loyal Customer': 1},
    'Type of Travel': {'Personal Travel': 0, 'Business travel': 1},
    'Class': {'Eco': 0, 'Eco Plus': 1, 'Business': 2},
}


def encode(df):
    out = df[FEATURES].copy()
    for c, m in CAT_MAPS.items():
        unknown = set(out[c].dropna().unique()) - set(m)
        assert not unknown, f'{c}: unmapped levels {unknown}'
        out[c] = out[c].map(m).astype('float32')
    return out


train, test, orig = encode(train_df), encode(test_df), encode(orig_df)
print(f'encoded train {train.shape}, test {test.shape}, original {orig.shape} | '
      f'NaN left for the trees: {int(train.isna().sum().sum())} train, {int(test.isna().sum().sum())} test')
display(train.head(3))

In [ ]:
PRIOR_ORIG = y_orig.mean()
FOLDS = list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(train, y))


def smoothed_rate(keys, target, prior, smooth=20):
    g = pd.DataFrame({'k': np.asarray(keys), 't': target}).groupby('k')['t'].agg(['sum', 'count'])
    return (g['sum'] + smooth * prior) / (g['count'] + smooth)


# --- orig_pred: LightGBM on the original survey (5-fold average) ---
p_orig = dict(objective='binary', metric='auc', n_estimators=5000, learning_rate=0.03, num_leaves=63,
              min_child_samples=20, colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
              random_state=SEED, n_jobs=-1, verbose=-1)
orig_pred_train, orig_pred_test, orig_oof = np.zeros(len(train)), np.zeros(len(test)), np.zeros(len(orig))
for tr, va in StratifiedKFold(5, shuffle=True, random_state=SEED).split(orig, y_orig):
    m = lgb.LGBMClassifier(**p_orig).fit(orig.iloc[tr], y_orig[tr], eval_set=[(orig.iloc[va], y_orig[va])],
                                         callbacks=[lgb.early_stopping(200, verbose=False)])
    orig_oof[va] = m.predict_proba(orig.iloc[va])[:, 1]
    orig_pred_train += m.predict_proba(train)[:, 1] / 5
    orig_pred_test += m.predict_proba(test)[:, 1] / 5
print(f'original-data model: AUC {roc_auc_score(y_orig, orig_oof):.4f} on its own labels, '
      f'{roc_auc_score(y, orig_pred_train):.4f} on the competition labels  <- the label-noise ceiling')

# --- static blocks: orig_pred, orig_te, freq ---
static = {}
for name, df in (('train', train), ('test', test)):
    block = {'orig_pred': orig_pred_train if name == 'train' else orig_pred_test}
    for c in FEATURES:
        block[f'orig_te_{c}'] = df[c].map(smoothed_rate(orig[c], y_orig, PRIOR_ORIG)).to_numpy()
    for c in CONT_COLS:
        block[f'freq_{c}'] = df[c].map(pd.concat([train[c], test[c]]).value_counts()).to_numpy(dtype='float64')
    static[name] = pd.concat([df, pd.DataFrame(block, index=df.index)], axis=1)

# --- te / te_pair keys: one integer per value combination, shared by train and test ---
TE_SPECS = [('Age',), ('Flight Distance',), ('Class', 'Flight Distance'), ('Type of Travel', 'Flight Distance'),
            ('Class', 'Age'), ('Type of Travel', 'Age'), ('Customer Type', 'Age')]
both = pd.concat([train, test], ignore_index=True)
TE_KEYS = {}
for cols in TE_SPECS:
    code_ = np.zeros(len(both), dtype=np.int64)
    for c in cols:
        k = pd.factorize(both[c], use_na_sentinel=False)[0]
        code_ = code_ * (k.max() + 1) + k
    TE_KEYS[cols] = (code_[:len(train)], code_[len(train):])
del both


def fold_te(tr, va):
    """te / te_pair for one fold: (training rows, validation rows, test rows), no row sees its own label."""
    prior, out = y[tr].mean(), ({}, {}, {})
    inner = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(tr, y[tr]))
    for cols in TE_SPECS:
        name = 'te_' + '_x_'.join(c.lower().replace(' ', '_') for c in cols)
        k_train, k_test = TE_KEYS[cols]
        k_tr, col = pd.Series(k_train[tr]), np.full(len(tr), np.nan)
        for a, b in inner:
            col[b] = k_tr.iloc[b].map(smoothed_rate(k_tr.iloc[a], y[tr][a], prior)).to_numpy()
        rate_ = smoothed_rate(k_tr, y[tr], prior)
        out[0][name] = col
        out[1][name] = pd.Series(k_train[va]).map(rate_).to_numpy()
        out[2][name] = pd.Series(k_test).map(rate_).to_numpy()
    return out


# Every fold's matrices are built once and shared by the members
FOLD_DATA = []
for tr, va in FOLDS:
    t_tr, t_va, t_te = fold_te(tr, va)
    FOLD_DATA.append(tuple(pd.concat([base, pd.DataFrame(t, index=base.index)], axis=1) for base, t in
                           ((static['train'].iloc[tr], t_tr), (static['train'].iloc[va], t_va), (static['test'], t_te))))
print('features per model:', FOLD_DATA[0][0].shape[1])

## 2. TabPFN-3.5 members

TabPFN does not train weights on our data: it reads a fold's training rows as its context and predicts the fold's
validation rows and the test set in one forward pass. Each member is one such fit per fold (and per seed); the test
predictions are averaged over the folds and seeds.

| member | inputs | columns |
|---|---|---|
| `pfn_enc` | everything above: raw columns, `orig_pred`, `orig_te`, `freq`, `te` / `te_pair` | 54 |
| `pfn_orig` | raw columns, `orig_pred`, `orig_te` (nothing learned from the competition labels) | 43 |

- `RUN` lists the members of the full run, `SEEDS` the seeds per fold.
- `CONTEXT` is the number of training rows per fit. `None` uses the whole training fold (about 560k rows, within
  TabPFN-3.5's limit of 1M); a number draws a stratified subsample, a different one per seed.
- With two GPUs the jobs (member × fold × seed) run two at a time, one per GPU.
- A fit that runs out of GPU memory is retried with half the context; the smoke run reports the context that fits.
- Each member is saved as soon as its folds finish, with `members.csv` for the meta-stack.

In [ ]:
ALL_COLS = list(FOLD_DATA[0][0].columns)
MEMBERS = {
    'pfn_enc':  ALL_COLS,
    'pfn_orig': FEATURES + ['orig_pred'] + [c for c in ALL_COLS if c.startswith('orig_te_')],
}
RUN = ['pfn_enc']           # members of the full run
SEEDS = [0]                 # one fit per fold and seed
CONTEXT = None              # training rows per fit; None = the whole training fold
PFN_KW = dict(n_estimators=4, fit_mode='fit_with_cache', ignore_pretraining_limits=True)
PRED_CHUNK = 50_000         # rows per predict_proba call
SMOKE_TEST_ROWS = 20_000
print({m: len(c) for m, c in MEMBERS.items()}, '| run:', RUN, '| seeds:', SEEDS, '| context:', CONTEXT or 'whole fold')

# Probe on 2k rows: downloads the weights once (this is where a missing licence token shows up),
# before two threads would try to download them at the same time
t0 = time.time()
probe = TabPFNClassifier(device='cuda:0', n_estimators=1).fit(
    FOLD_DATA[0][0][ALL_COLS].to_numpy('float32')[:2000], y[FOLDS[0][0]][:2000])
print(f'TabPFN ready in {time.time() - t0:.0f}s')
del probe; gc.collect(); torch.cuda.empty_cache()


def predict(m, X):
    return np.concatenate([m.predict_proba(X[i:i + PRED_CHUNK])[:, 1] for i in range(0, len(X), PRED_CHUNK)])


def is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or 'out of memory' in str(e).lower()


def run_job(member, k, seed, gpu, n_test=None):
    """One TabPFN fit on fold k: validation and test probabilities, the context used, timings and peak memory."""
    tr, va = FOLDS[k]
    X_tr, X_va, X_te = (X[MEMBERS[member]].to_numpy('float32') for X in FOLD_DATA[k])
    X_te = X_te[:n_test] if n_test else X_te
    context = min(CONTEXT or len(tr), len(tr))
    while True:
        idx = (np.arange(len(tr)) if context == len(tr) else
               train_test_split(np.arange(len(tr)), train_size=context, stratify=y[tr], random_state=seed)[0])
        m = None
        try:
            torch.cuda.reset_peak_memory_stats(gpu)
            t0 = time.time()
            m = TabPFNClassifier(device=f'cuda:{gpu}', random_state=seed, **PFN_KW).fit(X_tr[idx], y[tr][idx])
            t1 = time.time(); p_va = predict(m, X_va)
            t2 = time.time(); p_te = predict(m, X_te)
            t3 = time.time()
            return dict(member=member, fold=k, seed=seed, gpu=gpu, context=len(idx), p_va=p_va, p_te=p_te,
                        auc=roc_auc_score(y[va], p_va), fit_s=t1 - t0, va_s=t2 - t1, te_s=t3 - t2,
                        n_test=len(X_te), mem_gb=torch.cuda.max_memory_allocated(gpu) / 1e9)
        except Exception as e:
            if not is_oom(e) or context <= 50_000:
                raise
            context //= 2
            print(f'  {member} fold {k} seed {seed}: out of GPU memory, retrying with context {context:,}', flush=True)
        finally:
            del m; gc.collect(); torch.cuda.empty_cache()


FREE = queue.Queue()
for g in range(N_GPU):
    FREE.put(g)


def on_free_gpu(job):
    g = FREE.get()
    try:
        with torch.cuda.device(g):
            r = run_job(*job, gpu=g, n_test=SMOKE_TEST_ROWS if SMOKE else None)
        print(f"  {r['member']} fold {r['fold']} seed {r['seed']} (GPU {g}): AUC {r['auc']:.6f} | "
              f"context {r['context']:,} | fit {r['fit_s'] / 60:.1f} min, validation {r['va_s'] / 60:.1f} min, "
              f"test {r['te_s'] / 60:.1f} min | peak {r['mem_gb']:.1f} GB", flush=True)
        return r
    finally:
        FREE.put(g)


def run_all(jobs):
    """Every job once, as many at a time as there are GPUs."""
    with ThreadPoolExecutor(N_GPU) as ex:
        return list(ex.map(on_free_gpu, jobs))

### Smoke run

One full-size fold per GPU for the first member in `RUN`, with the test set cut to 20k rows. It checks the token,
the GPUs and the memory, and projects the full run from the measured times (Kaggle stops a run at 12 hours).

In [ ]:
if SMOKE:
    t0 = time.time()
    res = run_all([(RUN[0], k, SEEDS[0]) for k in range(N_GPU)])
    per_job = max(r['fit_s'] + r['va_s'] + r['te_s'] / r['n_test'] * len(test) for r in res)
    n_jobs = len(RUN) * N_SPLITS * len(SEEDS)
    print(f'\n{N_GPU} fold(s) in parallel: {(time.time() - t0) / 60:.1f} min')
    print(f'one full job (fit + {len(FOLDS[0][1]):,} validation + {len(test):,} test rows): about {per_job / 60:.0f} min')
    print(f'full run: {len(RUN)} member(s) x {N_SPLITS} folds x {len(SEEDS)} seed(s) = {n_jobs} jobs on {N_GPU} GPU(s), '
          f'about {-(-n_jobs // N_GPU) * per_job / 3600:.1f} h')
    print(f'context that fits: {min(r["context"] for r in res):,} rows | peak GPU memory '
          f'{max(r["mem_gb"] for r in res):.1f} GB of {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB')

### Full run

Every member in `RUN` is saved as soon as its folds finish, and a member already saved in `/kaggle/working` is
skipped. `members.csv` lists every saved member and is what the meta-stack notebook looks for.

In [ ]:
def write_card():
    done = sorted(f[4:-4] for f in os.listdir(OUT) if f.startswith('oof_pfn_') and f.endswith('.npy'))
    card = pd.DataFrame({'member': done, 'oof_auc': [roc_auc_score(y, np.load(f'{OUT}/oof_{m}.npy')) for m in done]})
    card.to_csv(f'{OUT}/members.csv', index=False)
    return card


if not SMOKE:
    for member in RUN:
        if os.path.exists(f'{OUT}/oof_{member}.npy'):
            print(f'{member}: already done'); continue
        t0 = time.time()
        res = run_all([(member, k, s) for k in range(N_SPLITS) for s in SEEDS])
        oof, pred = np.zeros(len(y)), np.zeros(len(test))
        for r in res:
            oof[FOLDS[r['fold']][1]] += r['p_va'] / len(SEEDS)
            pred += r['p_te'] / (N_SPLITS * len(SEEDS))
        print(f'{member:10s} OOF AUC {roc_auc_score(y, oof):.6f}   ({(time.time() - t0) / 60:.0f} min)', flush=True)
        np.save(f'{OUT}/oof_{member}.npy', oof)
        np.save(f'{OUT}/test_{member}.npy', pred)
        display(write_card().round(6))
        del res; gc.collect()